# Read-only end-to-end validation

Run after all four ingestion activities and the Silver and Gold notebooks complete. This notebook checks the required metadata, duplicate content versions, SCD Type 2 current rows and validity intervals, Gold resource keys, Raw files and sidecars, and audit/watermark tables. It does not modify project data.

In [ ]:
RAISE_ON_FAILURE = False

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")

def table_exists(table_name):
    return spark.catalog.tableExists(table_name)

results = []
for resource_type in RESOURCE_TYPES:
    bronze_table = f"bronze.{resource_type}"
    silver_table = f"silver.{resource_type}"
    gold_table = f"gold.{resource_type}"
    required_tables = [bronze_table, silver_table, gold_table]
    missing_tables = [name for name in required_tables if not table_exists(name)]

    if missing_tables:
        results.append({
            "resource_type": resource_type,
            "status": "FAIL",
            "reason": "Missing tables: " + ", ".join(missing_tables)
        })
        continue

    bronze = spark.table(bronze_table)
    silver = spark.table(silver_table)
    gold = spark.table(gold_table)

    required_columns = [
        "resource_id", "resource_json", "content_hash",
        "extraction_timestamp", "api_url_or_params", "saved_timestamp"
    ]
    missing_columns = [name for name in required_columns if name not in bronze.columns]
    bronze_null_count = 0
    if not missing_columns:
        for column_name in required_columns:
            bronze_null_count += bronze.filter(F.col(column_name).isNull()).count()

    duplicate_bronze_versions = (
        bronze.groupBy("resource_type", "resource_id", "content_hash")
        .count().filter(F.col("count") > 1).count()
    )

    invalid_current_ids = (
        silver.groupBy("resource_id")
        .agg(F.sum(F.col("is_current").cast("int")).alias("current_count"))
        .filter(F.col("current_count") != 1).count()
    )
    invalid_intervals = silver.filter(
        F.col("valid_to").isNotNull()
        & (F.col("valid_from").isNull() | (F.col("valid_from") >= F.col("valid_to")))
    ).count()
    resource_ids_with_versions = (
        silver.groupBy("resource_id").count().filter(F.col("count") > 1).count()
    )

    key_column = {
        "Patient": "patient_id",
        "Encounter": "encounter_id",
        "Observation": "observation_id",
        "Condition": "condition_id"
    }[resource_type]
    gold_missing_ids = gold.filter(F.col(key_column).isNull()).count()
    gold_duplicate_ids = (
        gold.groupBy(key_column).count()
        .filter(F.col(key_column).isNotNull() & (F.col("count") > 1)).count()
    )

    raw_dir = RAW_ROOT / f"resource={resource_type}"
    payload_files = []
    sidecar_files = []
    if raw_dir.exists():
        payload_files = [
            file for file in raw_dir.glob("extraction_date=*/run_id=*/page_*.json")
            if not file.name.endswith(".metadata.json")
        ]
        sidecar_files = list(raw_dir.glob("extraction_date=*/run_id=*/page_*.metadata.json"))

    passed = (
        not missing_columns
        and bronze_null_count == 0
        and duplicate_bronze_versions == 0
        and invalid_current_ids == 0
        and invalid_intervals == 0
        and gold_missing_ids == 0
        and gold_duplicate_ids == 0
        and len(payload_files) > 0
        and len(payload_files) == len(sidecar_files)
    )
    results.append({
        "resource_type": resource_type,
        "bronze_rows": bronze.count(),
        "silver_versions": silver.count(),
        "silver_ids_with_multiple_versions": resource_ids_with_versions,
        "gold_rows": gold.count(),
        "bronze_missing_required_columns": len(missing_columns),
        "bronze_required_null_values": bronze_null_count,
        "bronze_duplicate_content_versions": duplicate_bronze_versions,
        "silver_ids_without_exactly_one_current_version": invalid_current_ids,
        "silver_invalid_intervals": invalid_intervals,
        "gold_missing_ids": gold_missing_ids,
        "gold_duplicate_ids": gold_duplicate_ids,
        "raw_payload_pages": len(payload_files),
        "raw_sidecar_files": len(sidecar_files),
        "status": "PASS" if passed else "FAIL"
    })

print("=== Per-resource validation ===")
display(spark.createDataFrame(results))

control_checks = []
for table_name in ["dbo.fhir_ingestion_audit", "dbo.fhir_ingestion_watermarks"]:
    exists = table_exists(table_name)
    control_checks.append({
        "table": table_name,
        "exists": exists,
        "rows": spark.table(table_name).count() if exists else 0
    })
print("=== Audit and watermark tables ===")
display(spark.createDataFrame(control_checks))

if table_exists("dbo.fhir_ingestion_audit"):
    print("=== Successful runs (run this again on separate days) ===")
    display(
        spark.table("dbo.fhir_ingestion_audit")
        .filter(F.col("run_status") == "SUCCESS")
        .groupBy("resource_type", "run_id")
        .agg(
            F.count("*").alias("audit_rows"),
            F.min("api_call_timestamp").alias("first_api_call_utc"),
            F.max("saved_timestamp").alias("last_saved_utc")
        )
        .orderBy("resource_type", "run_id")
    )

overall_pass = (
    len(results) == 4
    and all(row.get("status") == "PASS" for row in results)
    and all(row["exists"] and row["rows"] > 0 for row in control_checks)
)
print("OVERALL:", "PASS" if overall_pass else "FAIL")
if str(RAISE_ON_FAILURE).strip().lower() in ("true", "1", "yes") and not overall_pass:
    raise RuntimeError("FHIR assessment validation failed. Review the detail outputs.")
